In [2]:
import numpy as np
import msicpe.san as san
from plotly import express as px
import scipy.signal as signal

# Partie 1 - Introduction à l’analyse spectrale

In [3]:
signal_dict1 = np.load('generator.npz') # signal_dict est un dictionnaire contenant les données contenues dans le fichier signal.npz
s = signal_dict1['s'] # extraction du signal utile
t = signal_dict1['t'] # extraction du vecteur temps associé au signal

nu1, sF1 = san.trans_fourier(s, t)

spectre1 = np.abs(sF1)

px.line(x = t, y = s, labels={"x":"Temps", "y":"Amplitude"}, title="generator.npz").show()
px.line(x = nu1, y = spectre1, labels={"x":"Fréquence", "y":"Amplitude"}, title="spectre generator.npz").show()

In [4]:
signal_dict1 = np.load('piano_G.npz') # signal_dict est un dictionnaire contenant les données contenues dans le fichier signal.npz
s = signal_dict1['s'] # extraction du signal utile
t = signal_dict1['t'] # extraction du vecteur temps associé au signal

nu2, sF2 = san.trans_fourier(s, t)

spectre2 = np.abs(sF2)

px.line(x = t, y = s, labels={"x":"Temps", "y":"Amplitude"}, title="piano_G.npz").show()
px.line(x = nu2, y = spectre2, labels={"x":"Fréquence", "y":"Amplitude"}, title="spectre piano_G.npz").show()

In [5]:
signal_dict1 = np.load('noise.npz') # signal_dict est un dictionnaire contenant les données contenues dans le fichier signal.npz
s = signal_dict1['s'] # extraction du signal utile
t = signal_dict1['t'] # extraction du vecteur temps associé au signal

nu3, sF3 = san.trans_fourier(s, t)

spectre3 = np.abs(sF3)

px.line(x = t, y = s, labels={"x":"Temps", "y":"Amplitude"}, title="noise.npz").show()
px.line(x = nu3, y = spectre3, labels={"x":"Fréquence", "y":"Amplitude"}, title="spectre noise.npz").show()

In [6]:
signal_dict1 = np.load('ecg_signal.npz') # signal_dict est un dictionnaire contenant les données contenues dans le fichier signal.npz
s = signal_dict1['s'] # extraction du signal utile
t = signal_dict1['t'] # extraction du vecteur temps associé au signal

nu4, sF4 = san.trans_fourier(s - np.mean(s), t)

spectre4 = np.abs(sF4)

px.line(x = t, y = s, labels={"x":"Temps", "y":"Amplitude"}, title="ecg_signal.npz").show()
px.line(x = nu4, y = spectre4, labels={"x":"Fréquence", "y":"Amplitude"}, title="spectre ecg_signal.npz").show()

# Partie 2 - Reconnaissance automatique de voyelle

### 2.1 Lecture des fichiers et affichages

In [7]:
dat = np.load('apprentissage/a_1.npz') # dat est un dictionnaire contenant les données contenues dans le fichier .npz
s = dat['s'] # extraction du signal utile
t = dat['t'] # extraction du vecteur temps associé au signal

nu, tf_s = san.trans_fourier(s, t)

px.line(x = t, y = s, labels={"x":"Temps", "y":"Amplitude"}, title="Signal temporel a_1").show()


In [8]:
px.line(x = nu, y = abs(tf_s), labels={"x":"Fréquence", "y":"Amplitude"}, title="Spectre du signal a_1").show()

Les différents pics correspondent aux harmoniques du signal a_1

In [9]:
px.line(x = nu, y = 20*np.log10(abs(tf_s)), labels={"x":"Fréquence", "y":"Amplitude en log base 10"}, title="Log(base 10) du spectre du signal a_1").show()

Le fait d'appliquer le log base 10 permet de normaliser le spectre indépendamment de l'énergie totale.


### 2.2 Apprentissage − Caractérisation des voyelles

In [10]:
H = 10 # Nombre d'harmoniques

# Fonction qui crée le vecteur cherché
def vecteur(nom_fichier):
    dat = np.load(nom_fichier)
    s = dat['s']
    s = s - np.mean(s)
    t = dat['t']

    nu1, tf_s = san.trans_fourier(s, t)
    spectre = np.abs(tf_s)
    spectre_normalise = spectre / np.max(spectre)
    f0 = san.detect_fondamentale(spectre_normalise, nu1)
    L_nu = [(k + 1) * f0 for k in range (H + 1)]
    F, A = san.detect_pics(np.log10(spectre_normalise + 1e-12), L_nu, nu1)
    return A , F


In [20]:
# Contruction des vecteurs moyens

voyelles = ["a", "e", "i", "o", "u"]
vecteurs_moyens = {}

for voy in voyelles:
    V = []
    for i in range(1, 6): 
        A, F = vecteur(f"apprentissage/{voy}_{i}.npz") 
        V.append(A)
    
    V = np.array(V)
    vecteurs_moyens[voy] = np.mean(V, axis=0)

    # Affichage des 5 vecteurs moyens
    print("Le vecteur moyen de la voyelle", voy, "est :", vecteurs_moyens[voy])


Le vecteur moyen de la voyelle a est : [-0.01886931 -0.13011764 -0.7128447  -0.5064393  -0.58431494 -0.32283798
 -0.48776332 -0.56146836 -0.72584087 -0.8873106  -0.61565626]
Le vecteur moyen de la voyelle e est : [-0.08077179 -0.08191578 -0.53042513 -0.2546302  -1.0167363  -1.3291765
 -1.4077471  -1.4436018  -1.694505   -1.4867724  -1.4959902 ]
Le vecteur moyen de la voyelle i est : [-0.03671918 -0.15251617 -1.1153077  -1.8446983  -2.3706837  -2.4162474
 -2.6475408  -2.932525   -3.001329   -3.1691613  -3.0951128 ]
Le vecteur moyen de la voyelle o est : [-0.23810712 -0.05430543 -0.0877648  -0.64764214 -1.2985758  -1.2492272
 -1.9535471  -2.3705332  -2.623577   -2.754801   -2.8251183 ]
Le vecteur moyen de la voyelle u est : [-0.00310862 -0.20122436 -1.1865972  -1.8682756  -2.1434228  -2.4252865
 -2.7985187  -2.88631    -2.8463056  -2.8614697  -2.5386345 ]


In [12]:
# Fonction pour calculer la distance euclidienne
def distance(v1, v2):
    return np.linalg.norm(v1 - v2)


In [13]:
# Fonction pour classifier un son
def classifier(nom_fichier):
    A_test, F_test = vecteur(nom_fichier)

    distances = {}
    for voyelle in vecteurs_moyens:
        distances[voyelle] = distance(A_test, vecteurs_moyens[voyelle])

    print("Distances :", distances)

    # Voyelle prédite
    return min(distances, key=distances.get)

### 2.3 Test de la méthode

In [14]:
print(classifier("test/a_6.npz"))
print(classifier("test/a_7.npz"))
print(classifier("test/e_6.npz"))
print(classifier("test/e_7.npz"))
print(classifier("test/i_6.npz"))
print(classifier("test/i_7.npz"))
print(classifier("test/o_6.npz"))
print(classifier("test/o_7.npz"))
print(classifier("test/u_6.npz"))
print(classifier("test/u_7.npz"))

Distances : {'a': np.float32(0.8126141), 'e': np.float32(2.4272182), 'i': np.float32(6.3004704), 'o': np.float32(4.514053), 'u': np.float32(5.940471)}
a
Distances : {'a': np.float32(1.0752746), 'e': np.float32(1.6230221), 'i': np.float32(5.0552974), 'o': np.float32(3.5971673), 'u': np.float32(4.691805)}
a
Distances : {'a': np.float32(2.3799076), 'e': np.float32(0.6410718), 'i': np.float32(4.2078824), 'o': np.float32(2.5818322), 'u': np.float32(3.8035657)}
e
Distances : {'a': np.float32(2.0116951), 'e': np.float32(0.6140176), 'i': np.float32(4.4513736), 'o': np.float32(2.6373389), 'u': np.float32(4.1018915)}
e
Distances : {'a': np.float32(6.1210465), 'e': np.float32(4.2905197), 'i': np.float32(0.8622076), 'o': np.float32(3.0252085), 'u': np.float32(0.85794795)}
u
Distances : {'a': np.float32(6.25542), 'e': np.float32(4.4451637), 'i': np.float32(0.9009539), 'o': np.float32(3.1323707), 'u': np.float32(1.0855045)}
i
Distances : {'a': np.float32(5.123429), 'e': np.float32(3.2495637), 'i': n

### 2.4  Application au cas d'étude

In [15]:
print (classifier("etude/signal_5.npz"))

Distances : {'a': np.float32(2.2619553), 'e': np.float32(1.2062625), 'i': np.float32(4.2617145), 'o': np.float32(2.5960517), 'u': np.float32(3.9234543)}
e


## Partie affichage pour la présentation

In [16]:
# Affichage du signal à classifier
dat_class = np.load("etude/signal_5.npz")
s_class = dat_class["s"]
t_class = dat_class["t"]
nu_class, tf_s_class = san.trans_fourier(s_class, t_class)

px.line(x = t_class, y = s_class, labels={"x":"Temps", "y":"Amplitude"}, title="Signal temporel à classifier").show()
px.line(x = nu_class, y = abs(tf_s_class), labels={"x":"Fréquence", "y":"Amplitude"}, title="Spectre du signal à classifier").show()

In [17]:
# Affichage des 5 spectres voulus
def affichage(voyelle):
    dat = np.load(f"apprentissage/{voyelle}_1.npz")
    s = dat["s"]
    t = dat["t"]
    nu, tf_s = san.trans_fourier(s, t)
    px.line(x = nu, y = np.abs(tf_s), labels={"x":"Fréquence", "y":"Amplitude"}, title=f"Spectre du signal {voyelle}_1").show()

voyelles = ['a', 'e', 'i', 'o', 'u']
for voy in voyelles:
    print(affichage(voy))

None


None


None


None


None
